In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, gzip, time
import numpy as np

LANGUAGES = ["en", "fr", "de"]

QWEN_SCORED_PATH = "/content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10/eval/dev_scored_epoch1_top10.json.gz"
NEMO_SCORED_PATH = "/content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10/eval/dev_scored_epoch2_top10.json.gz"

print("Checking files...", flush=True)
print("Qwen exists:", os.path.exists(QWEN_SCORED_PATH), QWEN_SCORED_PATH, flush=True)
print("Nemo exists:", os.path.exists(NEMO_SCORED_PATH), NEMO_SCORED_PATH, flush=True)

print("Qwen size MB:", os.path.getsize(QWEN_SCORED_PATH) / 1024 / 1024, flush=True)
print("Nemo size MB:", os.path.getsize(NEMO_SCORED_PATH) / 1024 / 1024, flush=True)

def load_json_gz(path, name):
    print(f"Loading {name}...", flush=True)
    t0 = time.time()
    with gzip.open(path, "rt", encoding="utf-8") as f:
        obj = json.load(f)
    print(f"Loaded {name} in {time.time() - t0:.1f}s", flush=True)
    return obj

qwen = load_json_gz(QWEN_SCORED_PATH, "Qwen")
nemo = load_json_gz(NEMO_SCORED_PATH, "Nemotron")

for lang in LANGUAGES:
    print(lang, "qwen samples:", len(qwen[lang]), "nemo samples:", len(nemo[lang]), flush=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking files...
Qwen exists: True /content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10/eval/dev_scored_epoch1_top10.json.gz
Nemo exists: True /content/drive/MyDrive/ct26_nemotron1b_full_finetune_listwise_top10/eval/dev_scored_epoch2_top10.json.gz
Qwen size MB: 29.13430404663086
Nemo size MB: 29.13987636566162
Loading Qwen...
Loaded Qwen in 2.3s
Loading Nemotron...
Loaded Nemotron in 2.3s
en qwen samples: 3905 nemo samples: 3905
fr qwen samples: 702 nemo samples: 702
de qwen samples: 386 nemo samples: 386


In [ ]:
def zscore(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

def compute_metrics_from_ranks(ranks):
    ranks = np.asarray(ranks)
    return {
        "MRR@1": float(np.mean([1.0 / r if r <= 1 else 0.0 for r in ranks])),
        "MRR@5": float(np.mean([1.0 / r if r <= 5 else 0.0 for r in ranks])),
        "MRR@10": float(np.mean([1.0 / r if r <= 10 else 0.0 for r in ranks])),
        "Recall@5": float((ranks <= 5).mean()),
        "Recall@10": float((ranks <= 10).mean()),
    }

def evaluate_ensemble(weights_by_lang):
    all_lang_metrics = {}

    for lang in LANGUAGES:
        ranks = []
        wd, wq, wn = weights_by_lang[lang]

        q_samples = qwen[lang]
        n_samples = nemo[lang]

        assert len(q_samples) == len(n_samples), lang

        for qs, ns in zip(q_samples, n_samples):
            assert str(qs["gold_id"]) == str(ns["gold_id"])

            gold_id = str(qs["gold_id"])
            candidate_ids = [str(x) for x in qs["candidate_ids"]]

            nemo_score_by_id = {
                str(cid): score
                for cid, score in zip(ns["candidate_ids"], ns["ce_scores"])
            }

            dense_scores = np.asarray(qs["dense_scores"], dtype=np.float32)
            qwen_scores = np.asarray(qs["ce_scores"], dtype=np.float32)
            nemo_scores = np.asarray([nemo_score_by_id[cid] for cid in candidate_ids], dtype=np.float32)

            final_scores = (
                wd * zscore(dense_scores)
                + wq * zscore(qwen_scores)
                + wn * zscore(nemo_scores)
            )

            order = np.argsort(-final_scores)
            reranked_ids = [candidate_ids[i] for i in order]

            rank = reranked_ids.index(gold_id) + 1 if gold_id in reranked_ids else 10000
            ranks.append(rank)

        all_lang_metrics[lang] = compute_metrics_from_ranks(ranks)

    avg = {}
    for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
        avg[key] = float(np.mean([all_lang_metrics[lang][key] for lang in LANGUAGES]))

    return all_lang_metrics, avg

best = None

# Start coarse. Much faster than 0.025.
grid = np.arange(0.0, 1.0001, 0.05)

total = 0
for wd in grid:
    for wq in grid:
        wn = 1.0 - wd - wq
        if wn >= -1e-9:
            total += 1

print("Total combinations:", total, flush=True)

done = 0
t0 = time.time()

for wd in grid:
    for wq in grid:
        wn = 1.0 - wd - wq
        if wn < -1e-9:
            continue

        weights = {
            lang: (float(wd), float(wq), float(wn))
            for lang in LANGUAGES
        }

        per_lang, avg = evaluate_ensemble(weights)

        if best is None or avg["MRR@5"] > best["avg"]["MRR@5"]:
            best = {
                "weights": weights,
                "per_lang": per_lang,
                "avg": avg,
            }
            print(
                f"New best: MRR@5={avg['MRR@5']:.6f} "
                f"wd={wd:.2f} wq={wq:.2f} wn={wn:.2f}",
                flush=True,
            )

        done += 1
        if done % 25 == 0:
            print(f"Progress: {done}/{total} | elapsed {time.time() - t0:.1f}s", flush=True)

print("\n===== BEST GLOBAL ENSEMBLE =====")
print("weights:", best["weights"]["en"])
print("avg:", best["avg"])
for lang in LANGUAGES:
    print(lang, best["per_lang"][lang])

Total combinations: 231
New best: MRR@5=0.720338 wd=0.00 wq=0.00 wn=1.00
New best: MRR@5=0.723907 wd=0.00 wq=0.05 wn=0.95
New best: MRR@5=0.727362 wd=0.00 wq=0.10 wn=0.90
New best: MRR@5=0.730278 wd=0.00 wq=0.15 wn=0.85
New best: MRR@5=0.731621 wd=0.00 wq=0.20 wn=0.80
New best: MRR@5=0.734308 wd=0.00 wq=0.25 wn=0.75
New best: MRR@5=0.736924 wd=0.00 wq=0.30 wn=0.70
New best: MRR@5=0.739566 wd=0.00 wq=0.35 wn=0.65
New best: MRR@5=0.740976 wd=0.00 wq=0.40 wn=0.60
New best: MRR@5=0.743427 wd=0.00 wq=0.45 wn=0.55
New best: MRR@5=0.745163 wd=0.00 wq=0.50 wn=0.50
New best: MRR@5=0.745246 wd=0.00 wq=0.55 wn=0.45
New best: MRR@5=0.747144 wd=0.00 wq=0.60 wn=0.40
Progress: 25/231 | elapsed 21.2s
New best: MRR@5=0.748197 wd=0.05 wq=0.50 wn=0.45
New best: MRR@5=0.749295 wd=0.05 wq=0.55 wn=0.40
Progress: 50/231 | elapsed 42.9s
New best: MRR@5=0.750166 wd=0.10 wq=0.45 wn=0.45
New best: MRR@5=0.751532 wd=0.10 wq=0.50 wn=0.40
New best: MRR@5=0.751658 wd=0.10 wq=0.60 wn=0.30
New best: MRR@5=0.751724 wd=

In [ ]:
# ============================
# PER-LANGUAGE WEIGHT SEARCH
# ============================

import time
import numpy as np

def evaluate_single_language(lang, wd, wq, wn):
    ranks = []

    q_samples = qwen[lang]
    n_samples = nemo[lang]

    assert len(q_samples) == len(n_samples), lang

    for qs, ns in zip(q_samples, n_samples):
        assert str(qs["gold_id"]) == str(ns["gold_id"])

        gold_id = str(qs["gold_id"])
        candidate_ids = [str(x) for x in qs["candidate_ids"]]

        nemo_score_by_id = {
            str(cid): score
            for cid, score in zip(ns["candidate_ids"], ns["ce_scores"])
        }

        dense_scores = np.asarray(qs["dense_scores"], dtype=np.float32)
        qwen_scores = np.asarray(qs["ce_scores"], dtype=np.float32)
        nemo_scores = np.asarray([nemo_score_by_id[cid] for cid in candidate_ids], dtype=np.float32)

        final_scores = (
            wd * zscore(dense_scores)
            + wq * zscore(qwen_scores)
            + wn * zscore(nemo_scores)
        )

        order = np.argsort(-final_scores)
        reranked_ids = [candidate_ids[i] for i in order]

        rank = reranked_ids.index(gold_id) + 1 if gold_id in reranked_ids else 10000
        ranks.append(rank)

    return compute_metrics_from_ranks(ranks)


# Coarse per-language search
grid = np.arange(0.0, 1.0001, 0.025)

best_lang_weights = {}
best_lang_metrics = {}

t0 = time.time()

for lang in LANGUAGES:
    print(f"\n===== Searching weights for {lang.upper()} =====", flush=True)

    best_for_lang = None
    total = 0
    done = 0

    for wd in grid:
        for wq in grid:
            wn = 1.0 - wd - wq
            if wn >= -1e-9:
                total += 1

    for wd in grid:
        for wq in grid:
            wn = 1.0 - wd - wq
            if wn < -1e-9:
                continue

            metrics = evaluate_single_language(
                lang=lang,
                wd=float(wd),
                wq=float(wq),
                wn=float(wn),
            )

            if best_for_lang is None or metrics["MRR@5"] > best_for_lang["metrics"]["MRR@5"]:
                best_for_lang = {
                    "weights": (float(wd), float(wq), float(wn)),
                    "metrics": metrics,
                }

                print(
                    f"New best {lang.upper()}: "
                    f"MRR@5={metrics['MRR@5']:.6f} "
                    f"wd={wd:.3f} wq={wq:.3f} wn={wn:.3f}",
                    flush=True,
                )

            done += 1
            if done % 100 == 0:
                print(f"{lang.upper()} progress: {done}/{total}", flush=True)

    best_lang_weights[lang] = best_for_lang["weights"]
    best_lang_metrics[lang] = best_for_lang["metrics"]

print("\n===== BEST PER-LANGUAGE WEIGHTS =====")
for lang in LANGUAGES:
    print(lang, best_lang_weights[lang], best_lang_metrics[lang])

# Evaluate final per-language ensemble average
final_avg = {}
for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
    final_avg[key] = float(np.mean([best_lang_metrics[lang][key] for lang in LANGUAGES]))

print("\n===== PER-LANGUAGE ENSEMBLE AVG =====")
print(final_avg)

print(f"\nElapsed: {time.time() - t0:.1f}s")


===== Searching weights for EN =====
New best EN: MRR@5=0.718152 wd=0.000 wq=0.000 wn=1.000
New best EN: MRR@5=0.718583 wd=0.000 wq=0.025 wn=0.975
New best EN: MRR@5=0.719676 wd=0.000 wq=0.050 wn=0.950
New best EN: MRR@5=0.722343 wd=0.000 wq=0.075 wn=0.925
New best EN: MRR@5=0.724392 wd=0.000 wq=0.100 wn=0.900
New best EN: MRR@5=0.726022 wd=0.000 wq=0.125 wn=0.875
New best EN: MRR@5=0.728267 wd=0.000 wq=0.150 wn=0.850
New best EN: MRR@5=0.730098 wd=0.000 wq=0.175 wn=0.825
New best EN: MRR@5=0.731481 wd=0.000 wq=0.200 wn=0.800
New best EN: MRR@5=0.732847 wd=0.000 wq=0.225 wn=0.775
New best EN: MRR@5=0.734691 wd=0.000 wq=0.250 wn=0.750
New best EN: MRR@5=0.736201 wd=0.000 wq=0.275 wn=0.725
New best EN: MRR@5=0.737542 wd=0.000 wq=0.300 wn=0.700
New best EN: MRR@5=0.738169 wd=0.000 wq=0.325 wn=0.675
New best EN: MRR@5=0.740388 wd=0.000 wq=0.350 wn=0.650
New best EN: MRR@5=0.741729 wd=0.000 wq=0.375 wn=0.625
New best EN: MRR@5=0.742113 wd=0.000 wq=0.400 wn=0.600
New best EN: MRR@5=0.743295

In [ ]:
# ============================
# LOCAL REFINE PER-LANGUAGE WEIGHTS
# ============================

refined_lang_weights = {}
refined_lang_metrics = {}

starting_weights = {
    "en": (0.350, 0.425, 0.225),
    "fr": (0.125, 0.550, 0.325),
    "de": (0.050, 0.850, 0.100),
}

for lang in LANGUAGES:
    base_wd, base_wq, base_wn = starting_weights[lang]

    print(f"\n===== Refining {lang.upper()} around {starting_weights[lang]} =====", flush=True)

    wd_values = np.arange(max(0, base_wd - 0.05), min(1, base_wd + 0.05) + 1e-9, 0.005)
    wq_values = np.arange(max(0, base_wq - 0.05), min(1, base_wq + 0.05) + 1e-9, 0.005)

    best_for_lang = {
        "weights": starting_weights[lang],
        "metrics": evaluate_single_language(lang, base_wd, base_wq, base_wn),
    }

    for wd in wd_values:
        for wq in wq_values:
            wn = 1.0 - wd - wq
            if wn < -1e-9:
                continue

            metrics = evaluate_single_language(
                lang=lang,
                wd=float(wd),
                wq=float(wq),
                wn=float(wn),
            )

            if metrics["MRR@5"] > best_for_lang["metrics"]["MRR@5"]:
                best_for_lang = {
                    "weights": (float(wd), float(wq), float(wn)),
                    "metrics": metrics,
                }

                print(
                    f"New refined best {lang.upper()}: "
                    f"MRR@5={metrics['MRR@5']:.6f} "
                    f"wd={wd:.3f} wq={wq:.3f} wn={wn:.3f}",
                    flush=True,
                )

    refined_lang_weights[lang] = best_for_lang["weights"]
    refined_lang_metrics[lang] = best_for_lang["metrics"]

print("\n===== BEST REFINED PER-LANGUAGE WEIGHTS =====")
for lang in LANGUAGES:
    print(lang, refined_lang_weights[lang], refined_lang_metrics[lang])

refined_avg = {}
for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
    refined_avg[key] = float(np.mean([refined_lang_metrics[lang][key] for lang in LANGUAGES]))

print("\n===== REFINED PER-LANGUAGE ENSEMBLE AVG =====")
print(refined_avg)


===== Refining EN around (0.35, 0.425, 0.225) =====
New refined best EN: MRR@5=0.761968 wd=0.310 wq=0.470 wn=0.220

===== Refining FR around (0.125, 0.55, 0.325) =====
New refined best FR: MRR@5=0.800997 wd=0.150 wq=0.560 wn=0.290

===== Refining DE around (0.05, 0.85, 0.1) =====
New refined best DE: MRR@5=0.700691 wd=0.035 wq=0.860 wn=0.105
New refined best DE: MRR@5=0.701986 wd=0.035 wq=0.865 wn=0.100
New refined best DE: MRR@5=0.702202 wd=0.040 wq=0.860 wn=0.100

===== BEST REFINED PER-LANGUAGE WEIGHTS =====
en (0.31, 0.4700000000000001, 0.21999999999999986) {'MRR@1': 0.7131882202304738, 'MRR@5': 0.7619675629534783, 'MRR@10': 0.7644909863219721, 'Recall@5': 0.83303457106274, 'Recall@10': 0.8509603072983355}
fr (0.15000000000000008, 0.56, 0.2899999999999998) {'MRR@1': 0.7621082621082621, 'MRR@5': 0.800997150997151, 'MRR@10': 0.8021740605073938, 'Recall@5': 0.8547008547008547, 'Recall@10': 0.8632478632478633}
de (0.04, 0.86, 0.09999999999999998) {'MRR@1': 0.6580310880829016, 'MRR@5':

In [ ]:
print("qwen loaded:", "qwen" in globals())
print("nemo loaded:", "nemo" in globals())
print("LANGUAGES:", LANGUAGES)

print("refined_avg exists:", "refined_avg" in globals())
if "refined_avg" in globals():
    print("Current refined score ensemble MRR@5:", refined_avg["MRR@5"])

print("refined_lang_weights exists:", "refined_lang_weights" in globals())
if "refined_lang_weights" in globals():
    print(refined_lang_weights)

qwen loaded: True
nemo loaded: True
LANGUAGES: ['en', 'fr', 'de']
refined_avg exists: True
Current refined score ensemble MRR@5: 0.7550555954964965
refined_lang_weights exists: True
{'en': (0.31, 0.4700000000000001, 0.21999999999999986), 'fr': (0.15000000000000008, 0.56, 0.2899999999999998), 'de': (0.04, 0.86, 0.09999999999999998)}


In [ ]:
CURRENT_SCORE_ENSEMBLE_MRR5 = refined_avg["MRR@5"]
print("Baseline to beat:", CURRENT_SCORE_ENSEMBLE_MRR5)

Baseline to beat: 0.7550555954964965


In [ ]:
# ============================================================
# RRF / RANK FUSION ENSEMBLE
# Safe to run in same notebook after score-ensemble experiments.
#
# Uses existing variables:
#   qwen
#   nemo
#   LANGUAGES
#   compute_metrics_from_ranks
#
# Does NOT overwrite:
#   best
#   best_lang_weights
#   refined_lang_weights
#   refined_lang_metrics
#   refined_avg
# ============================================================

import numpy as np
import time

print("===== RRF / RANK FUSION START =====", flush=True)

# ----------------------------
# Baseline score ensemble
# ----------------------------
if "refined_avg" in globals():
    CURRENT_SCORE_ENSEMBLE_MRR5 = refined_avg["MRR@5"]
else:
    CURRENT_SCORE_ENSEMBLE_MRR5 = 0.7550555954964965

print("Score ensemble baseline MRR@5:", CURRENT_SCORE_ENSEMBLE_MRR5, flush=True)


# ----------------------------
# Helpers
# ----------------------------
def rrf_ranks_from_scores(scores):
    """
    Highest score gets rank 1.
    Returns rank per candidate index.
    """
    scores = np.asarray(scores, dtype=np.float32)
    order = np.argsort(-scores)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks.astype(np.float32)


def rrf_build_cache():
    """
    Precompute dense/qwen/nemo rank arrays for each query.
    This makes the RRF sweep much faster.
    """
    cache = {}

    for lang in LANGUAGES:
        print(f"Building RRF cache for {lang.upper()}...", flush=True)

        q_samples = qwen[lang]
        n_samples = nemo[lang]

        assert len(q_samples) == len(n_samples), f"Length mismatch for {lang}"

        rows = []

        for qs, ns in zip(q_samples, n_samples):
            assert str(qs["gold_id"]) == str(ns["gold_id"]), f"Gold mismatch in {lang}"

            gold_id = str(qs["gold_id"])
            candidate_ids = [str(x) for x in qs["candidate_ids"]]

            nemo_score_by_id = {
                str(cid): score
                for cid, score in zip(ns["candidate_ids"], ns["ce_scores"])
            }

            if not all(cid in nemo_score_by_id for cid in candidate_ids):
                raise ValueError(f"Candidate mismatch in {lang}")

            dense_scores = np.asarray(qs["dense_scores"], dtype=np.float32)
            qwen_scores = np.asarray(qs["ce_scores"], dtype=np.float32)
            nemo_scores = np.asarray(
                [nemo_score_by_id[cid] for cid in candidate_ids],
                dtype=np.float32,
            )

            rows.append({
                "gold_index": candidate_ids.index(gold_id) if gold_id in candidate_ids else -1,
                "dense_ranks": rrf_ranks_from_scores(dense_scores),
                "qwen_ranks": rrf_ranks_from_scores(qwen_scores),
                "nemo_ranks": rrf_ranks_from_scores(nemo_scores),
            })

        cache[lang] = rows
        print(f"Cached {len(rows)} rows for {lang.upper()}", flush=True)

    return cache


def rrf_eval_single_language(lang, wd, wq, wn, k):
    ranks = []

    for row in rrf_cache[lang]:
        gold_index = row["gold_index"]

        if gold_index < 0:
            ranks.append(10000)
            continue

        final_scores = (
            wd / (k + row["dense_ranks"])
            + wq / (k + row["qwen_ranks"])
            + wn / (k + row["nemo_ranks"])
        )

        order = np.argsort(-final_scores)
        rank = int(np.where(order == gold_index)[0][0]) + 1
        ranks.append(rank)

    return compute_metrics_from_ranks(ranks)


def rrf_eval_multilingual(weights_by_lang, k_by_lang):
    per_lang = {}

    for lang in LANGUAGES:
        wd, wq, wn = weights_by_lang[lang]
        k = k_by_lang[lang]

        per_lang[lang] = rrf_eval_single_language(
            lang=lang,
            wd=wd,
            wq=wq,
            wn=wn,
            k=k,
        )

    avg = {}
    for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
        avg[key] = float(np.mean([per_lang[lang][key] for lang in LANGUAGES]))

    return per_lang, avg


# ----------------------------
# Build cache
# ----------------------------
t0_all = time.time()
rrf_cache = rrf_build_cache()


# ----------------------------
# Global RRF search
# ----------------------------
print("\n===== GLOBAL RRF SEARCH =====", flush=True)

rrf_grid = np.arange(0.0, 1.0001, 0.025)

# Since reranking is only top-10, small k values are useful.
rrf_k_values = [1, 2, 3, 5, 10, 20, 40, 60]

rrf_best_global = None

rrf_total_global = 0
for k in rrf_k_values:
    for wd in rrf_grid:
        for wq in rrf_grid:
            wn = 1.0 - wd - wq
            if wn >= -1e-9:
                rrf_total_global += 1

print("Total global RRF combinations:", rrf_total_global, flush=True)

rrf_done = 0
t0 = time.time()

for k in rrf_k_values:
    for wd in rrf_grid:
        for wq in rrf_grid:
            wn = 1.0 - wd - wq
            if wn < -1e-9:
                continue

            weights_by_lang = {
                lang: (float(wd), float(wq), float(wn))
                for lang in LANGUAGES
            }

            k_by_lang = {
                lang: k
                for lang in LANGUAGES
            }

            per_lang, avg = rrf_eval_multilingual(weights_by_lang, k_by_lang)

            if rrf_best_global is None or avg["MRR@5"] > rrf_best_global["avg"]["MRR@5"]:
                rrf_best_global = {
                    "k": k,
                    "weights": weights_by_lang,
                    "per_lang": per_lang,
                    "avg": avg,
                }

                print(
                    f"New best GLOBAL RRF: "
                    f"MRR@5={avg['MRR@5']:.6f} "
                    f"k={k} wd={wd:.3f} wq={wq:.3f} wn={wn:.3f}",
                    flush=True,
                )

            rrf_done += 1
            if rrf_done % 500 == 0:
                print(
                    f"Global progress: {rrf_done}/{rrf_total_global} "
                    f"| elapsed {time.time() - t0:.1f}s",
                    flush=True,
                )

print("\n===== BEST GLOBAL RRF =====")
print("k:", rrf_best_global["k"])
print("weights:", rrf_best_global["weights"]["en"])
print("avg:", rrf_best_global["avg"])
for lang in LANGUAGES:
    print(lang, rrf_best_global["per_lang"][lang])


# ----------------------------
# Per-language RRF search
# ----------------------------
print("\n===== PER-LANGUAGE RRF SEARCH =====", flush=True)

rrf_best_lang_weights = {}
rrf_best_lang_metrics = {}
rrf_best_lang_k = {}

t0 = time.time()

for lang in LANGUAGES:
    print(f"\n----- Searching RRF for {lang.upper()} -----", flush=True)

    best_for_lang = None
    done_lang = 0

    for k in rrf_k_values:
        for wd in rrf_grid:
            for wq in rrf_grid:
                wn = 1.0 - wd - wq
                if wn < -1e-9:
                    continue

                metrics = rrf_eval_single_language(
                    lang=lang,
                    wd=float(wd),
                    wq=float(wq),
                    wn=float(wn),
                    k=k,
                )

                if best_for_lang is None or metrics["MRR@5"] > best_for_lang["metrics"]["MRR@5"]:
                    best_for_lang = {
                        "k": k,
                        "weights": (float(wd), float(wq), float(wn)),
                        "metrics": metrics,
                    }

                    print(
                        f"New best {lang.upper()} RRF: "
                        f"MRR@5={metrics['MRR@5']:.6f} "
                        f"k={k} wd={wd:.3f} wq={wq:.3f} wn={wn:.3f}",
                        flush=True,
                    )

                done_lang += 1
                if done_lang % 500 == 0:
                    print(f"{lang.upper()} progress: {done_lang}", flush=True)

    rrf_best_lang_k[lang] = best_for_lang["k"]
    rrf_best_lang_weights[lang] = best_for_lang["weights"]
    rrf_best_lang_metrics[lang] = best_for_lang["metrics"]

print("\n===== BEST PER-LANGUAGE RRF =====")
for lang in LANGUAGES:
    print(
        lang,
        "k=", rrf_best_lang_k[lang],
        "weights=", rrf_best_lang_weights[lang],
        "metrics=", rrf_best_lang_metrics[lang],
    )

rrf_lang_avg = {}
for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
    rrf_lang_avg[key] = float(np.mean([rrf_best_lang_metrics[lang][key] for lang in LANGUAGES]))

print("\n===== PER-LANGUAGE RRF AVG =====")
print(rrf_lang_avg)


# ----------------------------
# Final comparison
# ----------------------------
print("\n===== FINAL RRF COMPARISON =====")
print("Score ensemble MRR@5:", CURRENT_SCORE_ENSEMBLE_MRR5)
print("Global RRF MRR@5:", rrf_best_global["avg"]["MRR@5"])
print("Per-language RRF MRR@5:", rrf_lang_avg["MRR@5"])

print("Global RRF delta:", rrf_best_global["avg"]["MRR@5"] - CURRENT_SCORE_ENSEMBLE_MRR5)
print("Per-language RRF delta:", rrf_lang_avg["MRR@5"] - CURRENT_SCORE_ENSEMBLE_MRR5)

if rrf_lang_avg["MRR@5"] > CURRENT_SCORE_ENSEMBLE_MRR5 and rrf_lang_avg["MRR@5"] >= rrf_best_global["avg"]["MRR@5"]:
    print("\nDECISION: Use PER-LANGUAGE RRF.")
elif rrf_best_global["avg"]["MRR@5"] > CURRENT_SCORE_ENSEMBLE_MRR5:
    print("\nDECISION: Use GLOBAL RRF.")
else:
    print("\nDECISION: Keep REFINED SCORE ENSEMBLE.")

print(f"\nTotal RRF elapsed: {time.time() - t0_all:.1f}s")
print("===== RRF / RANK FUSION DONE =====")

===== RRF / RANK FUSION START =====
Score ensemble baseline MRR@5: 0.7550555954964965
Building RRF cache for EN...
Cached 3905 rows for EN
Building RRF cache for FR...
Cached 702 rows for FR
Building RRF cache for DE...
Cached 386 rows for DE

===== GLOBAL RRF SEARCH =====
Total global RRF combinations: 6888
New best GLOBAL RRF: MRR@5=0.720338 k=1 wd=0.000 wq=0.000 wn=1.000
New best GLOBAL RRF: MRR@5=0.720655 k=1 wd=0.000 wq=0.075 wn=0.925
New best GLOBAL RRF: MRR@5=0.721049 k=1 wd=0.000 wq=0.100 wn=0.900
New best GLOBAL RRF: MRR@5=0.721688 k=1 wd=0.000 wq=0.125 wn=0.875
New best GLOBAL RRF: MRR@5=0.722027 k=1 wd=0.000 wq=0.150 wn=0.850
New best GLOBAL RRF: MRR@5=0.722868 k=1 wd=0.000 wq=0.175 wn=0.825
New best GLOBAL RRF: MRR@5=0.724076 k=1 wd=0.000 wq=0.200 wn=0.800
New best GLOBAL RRF: MRR@5=0.724605 k=1 wd=0.000 wq=0.225 wn=0.775
New best GLOBAL RRF: MRR@5=0.725548 k=1 wd=0.000 wq=0.250 wn=0.750
New best GLOBAL RRF: MRR@5=0.726379 k=1 wd=0.000 wq=0.275 wn=0.725
New best GLOBAL RRF:

In [ ]:
# ============================================================
# SOFTMAX / LOG-PROBABILITY FUSION
#
# Formula:
#   final_score =
#       wd * log_softmax(dense_scores / Td)
#     + wq * log_softmax(qwen_scores  / Tq)
#     + wn * log_softmax(nemo_scores  / Tn)
#
# This keeps score magnitude, but calibrates each model per query.
#
# Safe to run in same notebook.
# Does NOT overwrite:
#   best
#   best_lang_weights
#   refined_lang_weights
#   refined_lang_metrics
#   refined_avg
# ============================================================

import numpy as np
import time

print("===== SOFTMAX / LOG-PROB FUSION START =====", flush=True)

# ----------------------------
# Baseline to beat
# ----------------------------
if "refined_avg" in globals():
    CURRENT_SCORE_ENSEMBLE_MRR5 = refined_avg["MRR@5"]
else:
    CURRENT_SCORE_ENSEMBLE_MRR5 = 0.7550555954964965

print("Score ensemble baseline MRR@5:", CURRENT_SCORE_ENSEMBLE_MRR5, flush=True)


# ----------------------------
# Helpers
# ----------------------------
def softmax_log_probs(scores, temperature):
    """
    scores: numpy array [N, K]
    returns log_softmax(scores / temperature), shape [N, K]
    """
    x = np.asarray(scores, dtype=np.float32) / float(temperature)
    x = x - np.max(x, axis=1, keepdims=True)
    log_den = np.log(np.sum(np.exp(x), axis=1, keepdims=True) + 1e-12)
    return x - log_den


def softmax_compute_metrics_from_score_matrix(final_scores, gold_indices):
    """
    final_scores: [N, K]
    gold_indices: [N], -1 if gold missing
    """
    ranks = []

    order = np.argsort(-final_scores, axis=1)

    for i, gold_idx in enumerate(gold_indices):
        if gold_idx < 0:
            ranks.append(10000)
            continue

        rank = int(np.where(order[i] == gold_idx)[0][0]) + 1
        ranks.append(rank)

    return compute_metrics_from_ranks(ranks)


def build_softmax_base_cache():
    """
    Aligns Qwen/Nemo candidates and stores dense/qwen/nemo score matrices.
    """
    cache = {}

    for lang in LANGUAGES:
        print(f"Building base cache for {lang.upper()}...", flush=True)

        q_samples = qwen[lang]
        n_samples = nemo[lang]

        assert len(q_samples) == len(n_samples), f"Length mismatch for {lang}"

        dense_rows = []
        qwen_rows = []
        nemo_rows = []
        gold_indices = []

        for qs, ns in zip(q_samples, n_samples):
            assert str(qs["gold_id"]) == str(ns["gold_id"]), f"Gold mismatch in {lang}"

            gold_id = str(qs["gold_id"])
            candidate_ids = [str(x) for x in qs["candidate_ids"]]

            nemo_score_by_id = {
                str(cid): score
                for cid, score in zip(ns["candidate_ids"], ns["ce_scores"])
            }

            if not all(cid in nemo_score_by_id for cid in candidate_ids):
                raise ValueError(f"Candidate mismatch in {lang}")

            dense_scores = np.asarray(qs["dense_scores"], dtype=np.float32)
            qwen_scores = np.asarray(qs["ce_scores"], dtype=np.float32)
            nemo_scores = np.asarray(
                [nemo_score_by_id[cid] for cid in candidate_ids],
                dtype=np.float32,
            )

            dense_rows.append(dense_scores)
            qwen_rows.append(qwen_scores)
            nemo_rows.append(nemo_scores)

            gold_indices.append(candidate_ids.index(gold_id) if gold_id in candidate_ids else -1)

        cache[lang] = {
            "dense_scores": np.vstack(dense_rows).astype(np.float32),
            "qwen_scores": np.vstack(qwen_rows).astype(np.float32),
            "nemo_scores": np.vstack(nemo_rows).astype(np.float32),
            "gold_indices": np.asarray(gold_indices, dtype=np.int32),
        }

        print(
            f"{lang.upper()} cached:",
            cache[lang]["dense_scores"].shape,
            flush=True,
        )

    return cache


def build_logprob_cache(base_cache, temp_values):
    """
    Precomputes log-softmax matrices for every temp.
    This makes the sweeps much faster.
    """
    logprob_cache = {}

    for lang in LANGUAGES:
        print(f"Precomputing log-probs for {lang.upper()}...", flush=True)

        logprob_cache[lang] = {
            "gold_indices": base_cache[lang]["gold_indices"],
            "dense": {},
            "qwen": {},
            "nemo": {},
        }

        for t in temp_values:
            logprob_cache[lang]["dense"][t] = softmax_log_probs(base_cache[lang]["dense_scores"], t)
            logprob_cache[lang]["qwen"][t] = softmax_log_probs(base_cache[lang]["qwen_scores"], t)
            logprob_cache[lang]["nemo"][t] = softmax_log_probs(base_cache[lang]["nemo_scores"], t)

    return logprob_cache


def eval_softmax_single_language(lang, wd, wq, wn, td, tq, tn):
    final_scores = (
        wd * softmax_logprob_cache[lang]["dense"][td]
        + wq * softmax_logprob_cache[lang]["qwen"][tq]
        + wn * softmax_logprob_cache[lang]["nemo"][tn]
    )

    return softmax_compute_metrics_from_score_matrix(
        final_scores=final_scores,
        gold_indices=softmax_logprob_cache[lang]["gold_indices"],
    )


def eval_softmax_multilingual(weights_by_lang, temps_by_lang):
    per_lang = {}

    for lang in LANGUAGES:
        wd, wq, wn = weights_by_lang[lang]
        td, tq, tn = temps_by_lang[lang]

        per_lang[lang] = eval_softmax_single_language(
            lang=lang,
            wd=wd,
            wq=wq,
            wn=wn,
            td=td,
            tq=tq,
            tn=tn,
        )

    avg = {}
    for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
        avg[key] = float(np.mean([per_lang[lang][key] for lang in LANGUAGES]))

    return per_lang, avg


# ----------------------------
# Build caches
# ----------------------------
t0_all = time.time()

softmax_base_cache = build_softmax_base_cache()

# Temperature grid.
# You can expand this later, but this is a strong first search.
softmax_temp_values = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0]

softmax_logprob_cache = build_logprob_cache(
    base_cache=softmax_base_cache,
    temp_values=softmax_temp_values,
)


# ----------------------------
# Stage 1: Search temperatures using your refined score-fusion weights
# ----------------------------
print("\n===== STAGE 1: TEMPERATURE SEARCH WITH CURRENT BEST WEIGHTS =====", flush=True)

if "refined_lang_weights" in globals():
    softmax_start_weights = refined_lang_weights
else:
    softmax_start_weights = {
        "en": (0.31, 0.47, 0.22),
        "fr": (0.15, 0.56, 0.29),
        "de": (0.04, 0.86, 0.10),
    }

print("Starting weights:", softmax_start_weights, flush=True)

softmax_best_temp_global = None

total_temp = len(softmax_temp_values) ** 3
done = 0
t0 = time.time()

for td in softmax_temp_values:
    for tq in softmax_temp_values:
        for tn in softmax_temp_values:
            temps_by_lang = {
                lang: (td, tq, tn)
                for lang in LANGUAGES
            }

            per_lang, avg = eval_softmax_multilingual(
                weights_by_lang=softmax_start_weights,
                temps_by_lang=temps_by_lang,
            )

            if softmax_best_temp_global is None or avg["MRR@5"] > softmax_best_temp_global["avg"]["MRR@5"]:
                softmax_best_temp_global = {
                    "temps": temps_by_lang,
                    "weights": softmax_start_weights,
                    "per_lang": per_lang,
                    "avg": avg,
                }

                print(
                    f"New best temp-global softmax: "
                    f"MRR@5={avg['MRR@5']:.6f} "
                    f"td={td} tq={tq} tn={tn}",
                    flush=True,
                )

            done += 1
            if done % 100 == 0:
                print(f"Temp search progress: {done}/{total_temp} | elapsed {time.time() - t0:.1f}s", flush=True)

print("\nBest global temps with fixed weights:")
print("temps:", softmax_best_temp_global["temps"]["en"])
print("avg:", softmax_best_temp_global["avg"])


# ----------------------------
# Stage 2: Per-language temperature search with current best weights
# ----------------------------
print("\n===== STAGE 2: PER-LANGUAGE TEMPERATURE SEARCH =====", flush=True)

softmax_best_lang_temps = {}
softmax_best_lang_temp_metrics = {}

for lang in LANGUAGES:
    print(f"\n----- Searching temps for {lang.upper()} -----", flush=True)

    wd, wq, wn = softmax_start_weights[lang]
    best_for_lang = None

    for td in softmax_temp_values:
        for tq in softmax_temp_values:
            for tn in softmax_temp_values:
                metrics = eval_softmax_single_language(
                    lang=lang,
                    wd=wd,
                    wq=wq,
                    wn=wn,
                    td=td,
                    tq=tq,
                    tn=tn,
                )

                if best_for_lang is None or metrics["MRR@5"] > best_for_lang["metrics"]["MRR@5"]:
                    best_for_lang = {
                        "temps": (td, tq, tn),
                        "weights": (wd, wq, wn),
                        "metrics": metrics,
                    }

                    print(
                        f"New best {lang.upper()} temps: "
                        f"MRR@5={metrics['MRR@5']:.6f} "
                        f"td={td} tq={tq} tn={tn}",
                        flush=True,
                    )

    softmax_best_lang_temps[lang] = best_for_lang["temps"]
    softmax_best_lang_temp_metrics[lang] = best_for_lang["metrics"]

softmax_temp_lang_avg = {}
for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
    softmax_temp_lang_avg[key] = float(np.mean([softmax_best_lang_temp_metrics[lang][key] for lang in LANGUAGES]))

print("\n===== BEST PER-LANGUAGE TEMPS WITH FIXED WEIGHTS =====")
for lang in LANGUAGES:
    print(lang, "temps=", softmax_best_lang_temps[lang], "weights=", softmax_start_weights[lang], "metrics=", softmax_best_lang_temp_metrics[lang])
print("avg:", softmax_temp_lang_avg)


# ----------------------------
# Stage 3: Per-language weight search using best per-language temps
# ----------------------------
print("\n===== STAGE 3: PER-LANGUAGE WEIGHT SEARCH WITH BEST TEMPS =====", flush=True)

softmax_weight_grid = np.arange(0.0, 1.0001, 0.025)

softmax_best_lang_weights = {}
softmax_best_lang_metrics = {}
softmax_best_lang_temps_final = {}

for lang in LANGUAGES:
    print(f"\n----- Searching weights for {lang.upper()} -----", flush=True)

    td, tq, tn = softmax_best_lang_temps[lang]
    best_for_lang = None

    total = 0
    for wd in softmax_weight_grid:
        for wq in softmax_weight_grid:
            wn = 1.0 - wd - wq
            if wn >= -1e-9:
                total += 1

    done = 0

    for wd in softmax_weight_grid:
        for wq in softmax_weight_grid:
            wn = 1.0 - wd - wq
            if wn < -1e-9:
                continue

            metrics = eval_softmax_single_language(
                lang=lang,
                wd=float(wd),
                wq=float(wq),
                wn=float(wn),
                td=td,
                tq=tq,
                tn=tn,
            )

            if best_for_lang is None or metrics["MRR@5"] > best_for_lang["metrics"]["MRR@5"]:
                best_for_lang = {
                    "weights": (float(wd), float(wq), float(wn)),
                    "temps": (td, tq, tn),
                    "metrics": metrics,
                }

                print(
                    f"New best {lang.upper()} softmax weights: "
                    f"MRR@5={metrics['MRR@5']:.6f} "
                    f"wd={wd:.3f} wq={wq:.3f} wn={wn:.3f} "
                    f"td={td} tq={tq} tn={tn}",
                    flush=True,
                )

            done += 1
            if done % 200 == 0:
                print(f"{lang.upper()} weight progress: {done}/{total}", flush=True)

    softmax_best_lang_weights[lang] = best_for_lang["weights"]
    softmax_best_lang_temps_final[lang] = best_for_lang["temps"]
    softmax_best_lang_metrics[lang] = best_for_lang["metrics"]


softmax_lang_avg = {}
for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
    softmax_lang_avg[key] = float(np.mean([softmax_best_lang_metrics[lang][key] for lang in LANGUAGES]))

print("\n===== BEST SOFTMAX PER-LANGUAGE RESULT =====")
for lang in LANGUAGES:
    print(
        lang,
        "weights=", softmax_best_lang_weights[lang],
        "temps=", softmax_best_lang_temps_final[lang],
        "metrics=", softmax_best_lang_metrics[lang],
    )

print("\nSoftmax/log-prob per-language avg:")
print(softmax_lang_avg)


# ----------------------------
# Stage 4: Local refine weights around best softmax weights
# ----------------------------
print("\n===== STAGE 4: LOCAL REFINE SOFTMAX WEIGHTS =====", flush=True)

softmax_refined_lang_weights = {}
softmax_refined_lang_temps = {}
softmax_refined_lang_metrics = {}

for lang in LANGUAGES:
    base_wd, base_wq, base_wn = softmax_best_lang_weights[lang]
    td, tq, tn = softmax_best_lang_temps_final[lang]

    print(
        f"\n----- Refining {lang.upper()} around weights={softmax_best_lang_weights[lang]} temps={(td, tq, tn)} -----",
        flush=True,
    )

    wd_values = np.arange(max(0, base_wd - 0.05), min(1, base_wd + 0.05) + 1e-9, 0.005)
    wq_values = np.arange(max(0, base_wq - 0.05), min(1, base_wq + 0.05) + 1e-9, 0.005)

    best_for_lang = {
        "weights": softmax_best_lang_weights[lang],
        "temps": softmax_best_lang_temps_final[lang],
        "metrics": softmax_best_lang_metrics[lang],
    }

    for wd in wd_values:
        for wq in wq_values:
            wn = 1.0 - wd - wq
            if wn < -1e-9:
                continue

            metrics = eval_softmax_single_language(
                lang=lang,
                wd=float(wd),
                wq=float(wq),
                wn=float(wn),
                td=td,
                tq=tq,
                tn=tn,
            )

            if metrics["MRR@5"] > best_for_lang["metrics"]["MRR@5"]:
                best_for_lang = {
                    "weights": (float(wd), float(wq), float(wn)),
                    "temps": (td, tq, tn),
                    "metrics": metrics,
                }

                print(
                    f"New refined {lang.upper()} softmax: "
                    f"MRR@5={metrics['MRR@5']:.6f} "
                    f"wd={wd:.3f} wq={wq:.3f} wn={wn:.3f} "
                    f"td={td} tq={tq} tn={tn}",
                    flush=True,
                )

    softmax_refined_lang_weights[lang] = best_for_lang["weights"]
    softmax_refined_lang_temps[lang] = best_for_lang["temps"]
    softmax_refined_lang_metrics[lang] = best_for_lang["metrics"]


softmax_refined_avg = {}
for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
    softmax_refined_avg[key] = float(np.mean([softmax_refined_lang_metrics[lang][key] for lang in LANGUAGES]))

print("\n===== BEST REFINED SOFTMAX / LOG-PROB RESULT =====")
for lang in LANGUAGES:
    print(
        lang,
        "weights=", softmax_refined_lang_weights[lang],
        "temps=", softmax_refined_lang_temps[lang],
        "metrics=", softmax_refined_lang_metrics[lang],
    )

print("\nRefined softmax/log-prob avg:")
print(softmax_refined_avg)


# ----------------------------
# Final comparison
# ----------------------------
print("\n===== FINAL SOFTMAX COMPARISON =====")
print("Previous refined z-score ensemble MRR@5:", CURRENT_SCORE_ENSEMBLE_MRR5)
print("Best softmax/log-prob MRR@5:", softmax_refined_avg["MRR@5"])
print("Delta:", softmax_refined_avg["MRR@5"] - CURRENT_SCORE_ENSEMBLE_MRR5)

if softmax_refined_avg["MRR@5"] > CURRENT_SCORE_ENSEMBLE_MRR5:
    print("\nDECISION: Use SOFTMAX / LOG-PROB FUSION.")
else:
    print("\nDECISION: Keep previous REFINED Z-SCORE ENSEMBLE.")

print(f"\nTotal elapsed: {time.time() - t0_all:.1f}s")
print("===== SOFTMAX / LOG-PROB FUSION DONE =====")

===== SOFTMAX / LOG-PROB FUSION START =====
Score ensemble baseline MRR@5: 0.7550555954964965
Building base cache for EN...
EN cached: (3905, 10)
Building base cache for FR...
FR cached: (702, 10)
Building base cache for DE...
DE cached: (386, 10)
Precomputing log-probs for EN...
Precomputing log-probs for FR...
Precomputing log-probs for DE...

===== STAGE 1: TEMPERATURE SEARCH WITH CURRENT BEST WEIGHTS =====
Starting weights: {'en': (0.31, 0.4700000000000001, 0.21999999999999986), 'fr': (0.15000000000000008, 0.56, 0.2899999999999998), 'de': (0.04, 0.86, 0.09999999999999998)}
New best temp-global softmax: MRR@5=0.747611 td=0.25 tq=0.25 tn=0.25
New best temp-global softmax: MRR@5=0.747897 td=0.25 tq=0.5 tn=0.5
New best temp-global softmax: MRR@5=0.748342 td=0.25 tq=0.5 tn=0.75
New best temp-global softmax: MRR@5=0.748525 td=0.25 tq=1.0 tn=1.0
New best temp-global softmax: MRR@5=0.748880 td=0.25 tq=1.0 tn=1.5
New best temp-global softmax: MRR@5=0.749042 td=0.25 tq=1.5 tn=1.5
New best te

In [ ]:
# ============================================================
# MINIMAL SUPERVISED STACKER / META-RANKER
# Requires existing: qwen, nemo, LANGUAGES, compute_metrics_from_ranks
# CPU only.
# ============================================================

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CURRENT_BEST = 0.7566674995807707

def zscore1d(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

def log_softmax1d(x, temp=1.0):
    x = np.asarray(x, dtype=np.float32) / temp
    x = x - x.max()
    return x - np.log(np.exp(x).sum() + 1e-12)

def ranks1d(x):
    order = np.argsort(-np.asarray(x))
    r = np.empty_like(order)
    r[order] = np.arange(1, len(x) + 1)
    return r.astype(np.float32)

def margin_features(scores):
    s = np.sort(np.asarray(scores, dtype=np.float32))[::-1]
    top1 = s[0]
    top2 = s[1] if len(s) > 1 else s[0]
    return top1 - top2

# Build pair-level training data
X, y, groups, row_meta = [], [], [], []
qid = 0

lang_to_id = {l: i for i, l in enumerate(LANGUAGES)}

for lang in LANGUAGES:
    for qs, ns in zip(qwen[lang], nemo[lang]):
        gold_id = str(qs["gold_id"])
        cids = [str(x) for x in qs["candidate_ids"]]

        nemo_by_id = {str(cid): score for cid, score in zip(ns["candidate_ids"], ns["ce_scores"])}

        dense = np.asarray(qs["dense_scores"], dtype=np.float32)
        qwen_s = np.asarray(qs["ce_scores"], dtype=np.float32)
        nemo_s = np.asarray([nemo_by_id[cid] for cid in cids], dtype=np.float32)

        dense_z = zscore1d(dense)
        qwen_z = zscore1d(qwen_s)
        nemo_z = zscore1d(nemo_s)

        dense_lp = log_softmax1d(dense, temp=0.25)
        qwen_lp = log_softmax1d(qwen_s, temp=4.0)
        nemo_lp = log_softmax1d(nemo_s, temp=4.0 if lang != "de" else 3.0)

        dense_r = ranks1d(dense)
        qwen_r = ranks1d(qwen_s)
        nemo_r = ranks1d(nemo_s)

        dense_margin = margin_features(dense)
        qwen_margin = margin_features(qwen_s)
        nemo_margin = margin_features(nemo_s)

        for i, cid in enumerate(cids):
            feats = [
                dense_z[i], qwen_z[i], nemo_z[i],
                dense_lp[i], qwen_lp[i], nemo_lp[i],
                dense_r[i], qwen_r[i], nemo_r[i],
                dense_z[i] - qwen_z[i],
                qwen_z[i] - nemo_z[i],
                dense_z[i] - nemo_z[i],
                float(dense_r[i] == 1),
                float(qwen_r[i] == 1),
                float(nemo_r[i] == 1),
                dense_margin,
                qwen_margin,
                nemo_margin,
                float(lang == "en"),
                float(lang == "fr"),
                float(lang == "de"),
            ]

            X.append(feats)
            y.append(1 if cid == gold_id else 0)
            groups.append(qid)
            row_meta.append((lang, qid, cid, gold_id))

        qid += 1

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int32)
groups = np.asarray(groups, dtype=np.int32)

print("X:", X.shape, "positives:", y.sum(), "queries:", len(set(groups)))

def eval_pair_scores(pair_scores, row_meta):
    by_query = {}

    for score, meta in zip(pair_scores, row_meta):
        lang, qid, cid, gold_id = meta
        by_query.setdefault((lang, qid), []).append((score, cid, gold_id))

    ranks_by_lang = {lang: [] for lang in LANGUAGES}

    for (lang, qid), rows in by_query.items():
        rows = sorted(rows, key=lambda x: -x[0])
        gold_id = rows[0][2]
        ranked_ids = [cid for _, cid, _ in rows]
        rank = ranked_ids.index(gold_id) + 1 if gold_id in ranked_ids else 10000
        ranks_by_lang[lang].append(rank)

    per_lang = {lang: compute_metrics_from_ranks(ranks_by_lang[lang]) for lang in LANGUAGES}
    avg = {
        k: float(np.mean([per_lang[lang][k] for lang in LANGUAGES]))
        for k in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]
    }
    return per_lang, avg

# ----------------------------
# 1) OOF evaluation
# ----------------------------
oof_scores = np.zeros(len(y), dtype=np.float32)

gkf = GroupKFold(n_splits=5)

for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
        )
    )

    clf.fit(X[tr], y[tr])
    oof_scores[va] = clf.predict_proba(X[va])[:, 1]

    _, fold_avg = eval_pair_scores(oof_scores[va], [row_meta[i] for i in va])
    print(f"Fold {fold} MRR@5:", fold_avg["MRR@5"])

oof_per_lang, oof_avg = eval_pair_scores(oof_scores, row_meta)

print("\n===== OOF STACKER RESULT =====")
print(oof_avg)
for lang in LANGUAGES:
    print(lang, oof_per_lang[lang])

# ----------------------------
# 2) In-sample dev-max result
# ----------------------------
stacker_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
    )
)

stacker_clf.fit(X, y)
insample_scores = stacker_clf.predict_proba(X)[:, 1]

insample_per_lang, insample_avg = eval_pair_scores(insample_scores, row_meta)

print("\n===== IN-SAMPLE STACKER RESULT =====")
print(insample_avg)
for lang in LANGUAGES:
    print(lang, insample_per_lang[lang])

print("\n===== COMPARISON =====")
print("Current best softmax:", CURRENT_BEST)
print("OOF stacker MRR@5:", oof_avg["MRR@5"], "delta:", oof_avg["MRR@5"] - CURRENT_BEST)
print("In-sample stacker MRR@5:", insample_avg["MRR@5"], "delta:", insample_avg["MRR@5"] - CURRENT_BEST)

if oof_avg["MRR@5"] > CURRENT_BEST:
    print("Decision: stacker looks genuinely promising.")
elif insample_avg["MRR@5"] > CURRENT_BEST:
    print("Decision: stacker can overfit/dev-max, but OOF did not beat current best.")
else:
    print("Decision: keep softmax/log-prob fusion.")

X: (49930, 21) positives: 4234 queries: 4993
Fold 1 MRR@5: 0.7392583304731642
Fold 2 MRR@5: 0.7783879291221215
Fold 3 MRR@5: 0.7210251863975268
Fold 4 MRR@5: 0.7405818750889172
Fold 5 MRR@5: 0.7766733735747819

===== OOF STACKER RESULT =====
{'MRR@1': 0.7076808392019173, 'MRR@5': 0.7511329260451513, 'MRR@10': 0.754578505156197, 'Recall@5': 0.8117084434922378, 'Recall@10': 0.8347878703202355}
en {'MRR@1': 0.717797695262484, 'MRR@5': 0.7640845070422535, 'MRR@10': 0.7663350811942361, 'Recall@5': 0.8348271446862996, 'Recall@10': 0.8509603072983355}
fr {'MRR@1': 0.7549857549857549, 'MRR@5': 0.7969135802469137, 'MRR@10': 0.7990062406729074, 'Recall@5': 0.8490028490028491, 'Recall@10': 0.8632478632478633}
de {'MRR@1': 0.6502590673575129, 'MRR@5': 0.6924006908462866, 'MRR@10': 0.6983941936014474, 'Recall@5': 0.7512953367875648, 'Recall@10': 0.7901554404145078}

===== IN-SAMPLE STACKER RESULT =====
{'MRR@1': 0.709579425325369, 'MRR@5': 0.7519228711228029, 'MRR@10': 0.7554339893831584, 'Recall@5

In [ ]:
# ============================================================
# XGBOOST LAMBDAMART STACKER / META-RANKER
# Requires existing from previous stacker cell:
#   X, y, groups, row_meta, LANGUAGES, compute_metrics_from_ranks
# CPU only.
# ============================================================

import sys, subprocess, importlib.util
import numpy as np
from sklearn.model_selection import GroupKFold

if importlib.util.find_spec("xgboost") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)

from xgboost import XGBRanker

CURRENT_BEST = 0.7566674995807707

def make_group_sizes(qids):
    _, counts = np.unique(qids, return_counts=True)
    return counts.tolist()

def sort_by_group(idx):
    idx = np.asarray(idx)
    return idx[np.argsort(groups[idx], kind="stable")]

def eval_pair_scores_xgb(pair_scores, row_meta_subset):
    by_query = {}

    for score, meta in zip(pair_scores, row_meta_subset):
        lang, qid, cid, gold_id = meta
        by_query.setdefault((lang, qid), []).append((float(score), cid, gold_id))

    ranks_by_lang = {lang: [] for lang in LANGUAGES}

    for (lang, qid), rows in by_query.items():
        rows = sorted(rows, key=lambda x: -x[0])
        gold_id = rows[0][2]
        ranked_ids = [cid for _, cid, _ in rows]
        rank = ranked_ids.index(gold_id) + 1 if gold_id in ranked_ids else 10000
        ranks_by_lang[lang].append(rank)

    per_lang = {
        lang: compute_metrics_from_ranks(ranks_by_lang[lang])
        for lang in LANGUAGES
    }

    avg = {
        k: float(np.mean([per_lang[lang][k] for lang in LANGUAGES]))
        for k in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]
    }

    return per_lang, avg

def make_ranker(seed=42):
    return XGBRanker(
        objective="rank:ndcg",
        eval_metric="ndcg@5",
        n_estimators=350,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=1.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5.0,
        reg_alpha=0.0,
        tree_method="hist",
        random_state=seed,
        n_jobs=-1,
    )

print("X:", X.shape, "positives:", int(y.sum()), "queries:", len(np.unique(groups)))

# ----------------------------
# 1) OOF XGBoost ranking eval
# ----------------------------
xgb_oof_scores = np.zeros(len(y), dtype=np.float32)

gkf = GroupKFold(n_splits=5)

for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
    tr_s = sort_by_group(tr)
    va_s = sort_by_group(va)

    xgb_ranker = make_ranker(seed=42 + fold)

    xgb_ranker.fit(
        X[tr_s],
        y[tr_s],
        group=make_group_sizes(groups[tr_s]),
        eval_set=[(X[va_s], y[va_s])],
        eval_group=[make_group_sizes(groups[va_s])],
        verbose=False,
    )

    xgb_oof_scores[va] = xgb_ranker.predict(X[va])

    _, fold_avg = eval_pair_scores_xgb(
        xgb_oof_scores[va],
        [row_meta[i] for i in va],
    )

    print(f"Fold {fold} MRR@5: {fold_avg['MRR@5']:.6f}")

xgb_oof_per_lang, xgb_oof_avg = eval_pair_scores_xgb(xgb_oof_scores, row_meta)

print("\n===== XGBOOST OOF RESULT =====")
print(xgb_oof_avg)
for lang in LANGUAGES:
    print(lang, xgb_oof_per_lang[lang])

# ----------------------------
# 2) In-sample dev-max result
# ----------------------------
all_idx = sort_by_group(np.arange(len(y)))

xgb_final_ranker = make_ranker(seed=123)

xgb_final_ranker.fit(
    X[all_idx],
    y[all_idx],
    group=make_group_sizes(groups[all_idx]),
    verbose=False,
)

xgb_insample_scores = xgb_final_ranker.predict(X)

xgb_insample_per_lang, xgb_insample_avg = eval_pair_scores_xgb(
    xgb_insample_scores,
    row_meta,
)

print("\n===== XGBOOST IN-SAMPLE RESULT =====")
print(xgb_insample_avg)
for lang in LANGUAGES:
    print(lang, xgb_insample_per_lang[lang])

print("\n===== COMPARISON =====")
print("Current best softmax:", CURRENT_BEST)
print("XGB OOF MRR@5:", xgb_oof_avg["MRR@5"], "delta:", xgb_oof_avg["MRR@5"] - CURRENT_BEST)
print("XGB in-sample MRR@5:", xgb_insample_avg["MRR@5"], "delta:", xgb_insample_avg["MRR@5"] - CURRENT_BEST)

if xgb_oof_avg["MRR@5"] > CURRENT_BEST:
    print("Decision: XGBoost looks genuinely promising.")
elif xgb_insample_avg["MRR@5"] > CURRENT_BEST:
    print("Decision: XGBoost can dev-max, but OOF did not beat current best.")
else:
    print("Decision: keep softmax/log-prob fusion.")

X: (49930, 21) positives: 4234 queries: 4993
Fold 1 MRR@5: 0.743614
Fold 2 MRR@5: 0.777700
Fold 3 MRR@5: 0.729826
Fold 4 MRR@5: 0.736699
Fold 5 MRR@5: 0.772708

===== XGBOOST OOF RESULT =====
{'MRR@1': 0.7078881414744608, 'MRR@5': 0.7520723655800321, 'MRR@10': 0.7547042976610981, 'Recall@5': 0.8163303452948055, 'Recall@10': 0.8347878703202355}
en {'MRR@1': 0.7201024327784891, 'MRR@5': 0.7655484421681606, 'MRR@10': 0.7679130337581042, 'Recall@5': 0.834314980793854, 'Recall@10': 0.8509603072983355}
fr {'MRR@1': 0.7507122507122507, 'MRR@5': 0.7939933523266857, 'MRR@10': 0.7957259304481525, 'Recall@5': 0.8504273504273504, 'Recall@10': 0.8632478632478633}
de {'MRR@1': 0.6528497409326425, 'MRR@5': 0.6966753022452503, 'MRR@10': 0.7004739287770376, 'Recall@5': 0.7642487046632125, 'Recall@10': 0.7901554404145078}

===== XGBOOST IN-SAMPLE RESULT =====
{'MRR@1': 0.726263302288017, 'MRR@5': 0.7648506738035795, 'MRR@10': 0.7667302465336555, 'Recall@5': 0.8221007638337178, 'Recall@10': 0.83478787032

In [ ]:
# ============================================================
# STRONGER XGBOOST STACKER:
# - Adds final softmax/log-prob fusion score as a feature
# - Tries rank:map and rank:ndcg
# - Uses regularized configs
# - Trains per-language OOF models
# - Blends XGB score with softmax score
#
# Requires existing:
#   X, y, groups, row_meta
#   qwen, nemo, LANGUAGES
#   compute_metrics_from_ranks
# ============================================================

import sys, subprocess, importlib.util
import numpy as np
from sklearn.model_selection import GroupKFold

if importlib.util.find_spec("xgboost") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)

from xgboost import XGBRanker

CURRENT_BEST = 0.7566674995807707

# ----------------------------
# Best softmax config found earlier
# ----------------------------
SOFTMAX_FINAL_WEIGHTS = {
    "en": (0.495, 0.380, 0.125),  # dense, qwen, nemo
    "fr": (0.300, 0.360, 0.340),
    "de": (0.475, 0.355, 0.170),
}

SOFTMAX_FINAL_TEMPS = {
    "en": (0.25, 4.0, 4.0),  # dense temp, qwen temp, nemo temp
    "fr": (0.25, 4.0, 4.0),
    "de": (0.25, 4.0, 3.0),
}

def log_softmax1d_local(x, temp):
    x = np.asarray(x, dtype=np.float32) / float(temp)
    x = x - x.max()
    return x - np.log(np.exp(x).sum() + 1e-12)

def build_softmax_pair_scores_in_x_order():
    scores = []

    for lang in LANGUAGES:
        wd, wq, wn = SOFTMAX_FINAL_WEIGHTS[lang]
        td, tq, tn = SOFTMAX_FINAL_TEMPS[lang]

        for qs, ns in zip(qwen[lang], nemo[lang]):
            cids = [str(x) for x in qs["candidate_ids"]]
            nemo_by_id = {str(cid): score for cid, score in zip(ns["candidate_ids"], ns["ce_scores"])}

            dense = np.asarray(qs["dense_scores"], dtype=np.float32)
            qwen_s = np.asarray(qs["ce_scores"], dtype=np.float32)
            nemo_s = np.asarray([nemo_by_id[cid] for cid in cids], dtype=np.float32)

            dense_lp = log_softmax1d_local(dense, td)
            qwen_lp = log_softmax1d_local(qwen_s, tq)
            nemo_lp = log_softmax1d_local(nemo_s, tn)

            final = wd * dense_lp + wq * qwen_lp + wn * nemo_lp
            scores.extend(final.tolist())

    return np.asarray(scores, dtype=np.float32)

def zscore_global(v):
    v = np.asarray(v, dtype=np.float32)
    return (v - v.mean()) / (v.std() + 1e-8)

def make_group_sizes(qids):
    _, counts = np.unique(qids, return_counts=True)
    return counts.tolist()

def sort_by_group(idx):
    idx = np.asarray(idx)
    return idx[np.argsort(groups[idx], kind="stable")]

def eval_pair_scores_any(pair_scores, row_meta_subset):
    by_query = {}

    for score, meta in zip(pair_scores, row_meta_subset):
        lang, qid, cid, gold_id = meta
        by_query.setdefault((lang, qid), []).append((float(score), cid, gold_id))

    ranks_by_lang = {lang: [] for lang in LANGUAGES}

    for (lang, qid), rows in by_query.items():
        rows = sorted(rows, key=lambda x: -x[0])
        gold_id = rows[0][2]
        ranked_ids = [cid for _, cid, _ in rows]
        rank = ranked_ids.index(gold_id) + 1 if gold_id in ranked_ids else 10000
        ranks_by_lang[lang].append(rank)

    per_lang = {
        lang: compute_metrics_from_ranks(ranks_by_lang[lang])
        for lang in LANGUAGES
    }

    avg = {
        k: float(np.mean([per_lang[lang][k] for lang in LANGUAGES]))
        for k in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]
    }

    return per_lang, avg

softmax_pair_scores = build_softmax_pair_scores_in_x_order()
assert len(softmax_pair_scores) == len(X), (len(softmax_pair_scores), len(X))

# Add softmax score as feature.
# This is important: XGBoost learns how to correct your current best system.
X_aug = np.column_stack([
    X,
    zscore_global(softmax_pair_scores),
]).astype(np.float32)

print("X original:", X.shape)
print("X augmented:", X_aug.shape)
print("Current best softmax MRR@5:", CURRENT_BEST)

# ----------------------------
# Candidate XGB configs
# Keep these regularized to reduce overfit.
# ----------------------------
xgb_configs = [
    {
        "name": "map_depth1_reg",
        "objective": "rank:map",
        "eval_metric": "map@5",
        "n_estimators": 120,
        "learning_rate": 0.03,
        "max_depth": 1,
        "min_child_weight": 8.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 30.0,
        "reg_alpha": 1.0,
    },
    {
        "name": "map_depth2_reg",
        "objective": "rank:map",
        "eval_metric": "map@5",
        "n_estimators": 180,
        "learning_rate": 0.025,
        "max_depth": 2,
        "min_child_weight": 8.0,
        "subsample": 0.80,
        "colsample_bytree": 0.85,
        "reg_lambda": 40.0,
        "reg_alpha": 1.0,
    },
    {
        "name": "ndcg_depth1_reg",
        "objective": "rank:ndcg",
        "eval_metric": "ndcg@5",
        "n_estimators": 120,
        "learning_rate": 0.03,
        "max_depth": 1,
        "min_child_weight": 8.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 30.0,
        "reg_alpha": 1.0,
    },
    {
        "name": "ndcg_depth2_reg",
        "objective": "rank:ndcg",
        "eval_metric": "ndcg@5",
        "n_estimators": 180,
        "learning_rate": 0.025,
        "max_depth": 2,
        "min_child_weight": 8.0,
        "subsample": 0.80,
        "colsample_bytree": 0.85,
        "reg_lambda": 40.0,
        "reg_alpha": 1.0,
    },
]

def make_ranker_from_config(cfg, seed):
    return XGBRanker(
        objective=cfg["objective"],
        eval_metric=cfg["eval_metric"],
        n_estimators=cfg["n_estimators"],
        learning_rate=cfg["learning_rate"],
        max_depth=cfg["max_depth"],
        min_child_weight=cfg["min_child_weight"],
        subsample=cfg["subsample"],
        colsample_bytree=cfg["colsample_bytree"],
        reg_lambda=cfg["reg_lambda"],
        reg_alpha=cfg["reg_alpha"],
        tree_method="hist",
        random_state=seed,
        n_jobs=-1,
    )

# ----------------------------
# Per-language OOF XGBoost
# ----------------------------
best_xgb_oof = None

for cfg in xgb_configs:
    print(f"\n===== Config: {cfg['name']} =====", flush=True)

    oof = np.zeros(len(y), dtype=np.float32)

    for lang in LANGUAGES:
        lang_idx = np.asarray([i for i, meta in enumerate(row_meta) if meta[0] == lang])
        lang_groups = groups[lang_idx]

        gkf = GroupKFold(n_splits=5)

        for fold, (tr_local, va_local) in enumerate(gkf.split(X_aug[lang_idx], y[lang_idx], lang_groups), 1):
            tr = sort_by_group(lang_idx[tr_local])
            va = sort_by_group(lang_idx[va_local])

            ranker = make_ranker_from_config(cfg, seed=1000 + fold)

            ranker.fit(
                X_aug[tr],
                y[tr],
                group=make_group_sizes(groups[tr]),
                eval_set=[(X_aug[va], y[va])],
                eval_group=[make_group_sizes(groups[va])],
                verbose=False,
            )

            oof[va] = ranker.predict(X_aug[va])

    # XGB alone
    per_lang_xgb, avg_xgb = eval_pair_scores_any(oof, row_meta)

    print("XGB OOF alone:", avg_xgb)

    # Blend XGB with softmax pair score
    best_blend = None

    for alpha in np.arange(0.0, 1.0001, 0.025):
        # alpha = weight on XGB
        blended = alpha * zscore_global(oof) + (1.0 - alpha) * zscore_global(softmax_pair_scores)

        per_lang_blend, avg_blend = eval_pair_scores_any(blended, row_meta)

        if best_blend is None or avg_blend["MRR@5"] > best_blend["avg"]["MRR@5"]:
            best_blend = {
                "alpha": float(alpha),
                "per_lang": per_lang_blend,
                "avg": avg_blend,
            }

    print(
        f"Best blend for {cfg['name']}: "
        f"MRR@5={best_blend['avg']['MRR@5']:.6f} "
        f"alpha_xgb={best_blend['alpha']:.3f}"
    )

    result = {
        "config": cfg,
        "oof_scores": oof,
        "xgb_avg": avg_xgb,
        "xgb_per_lang": per_lang_xgb,
        "blend": best_blend,
    }

    if best_xgb_oof is None or best_blend["avg"]["MRR@5"] > best_xgb_oof["blend"]["avg"]["MRR@5"]:
        best_xgb_oof = result

print("\n===== BEST REGULARIZED PER-LANGUAGE XGB OOF/BLEND =====")
print("config:", best_xgb_oof["config"]["name"])
print("XGB alone:", best_xgb_oof["xgb_avg"])
print("Best blend alpha_xgb:", best_xgb_oof["blend"]["alpha"])
print("Best blend avg:", best_xgb_oof["blend"]["avg"])
for lang in LANGUAGES:
    print(lang, best_xgb_oof["blend"]["per_lang"][lang])

print("\n===== COMPARISON =====")
print("Current best softmax:", CURRENT_BEST)
print("Best XGB blend OOF:", best_xgb_oof["blend"]["avg"]["MRR@5"])
print("Delta:", best_xgb_oof["blend"]["avg"]["MRR@5"] - CURRENT_BEST)

if best_xgb_oof["blend"]["avg"]["MRR@5"] > CURRENT_BEST:
    print("Decision: Use XGB+softmax blend.")
else:
    print("Decision: Keep softmax/log-prob fusion.")

X original: (49930, 21)
X augmented: (49930, 22)
Current best softmax MRR@5: 0.7566674995807707

===== Config: map_depth1_reg =====
XGB OOF alone: {'MRR@1': 0.700315197941022, 'MRR@5': 0.7486529676621059, 'MRR@10': 0.7512956874104203, 'Recall@5': 0.8159042911603178, 'Recall@10': 0.8347878703202355}
Best blend for map_depth1_reg: MRR@5=0.756667 alpha_xgb=0.000

===== Config: map_depth2_reg =====
XGB OOF alone: {'MRR@1': 0.701447036444239, 'MRR@5': 0.7490876563881258, 'MRR@10': 0.7520960358016991, 'Recall@5': 0.813530871228699, 'Recall@10': 0.8347878703202355}
Best blend for map_depth2_reg: MRR@5=0.756667 alpha_xgb=0.000

===== Config: ndcg_depth1_reg =====
XGB OOF alone: {'MRR@1': 0.6994516400826455, 'MRR@5': 0.7487143338722069, 'MRR@10': 0.7514432464460777, 'Recall@5': 0.815818930511577, 'Recall@10': 0.8347878703202355}
Best blend for ndcg_depth1_reg: MRR@5=0.756667 alpha_xgb=0.000

===== Config: ndcg_depth2_reg =====
XGB OOF alone: {'MRR@1': 0.7028227581950611, 'MRR@5': 0.749564277232